In [3]:
# Create empty list to store flattened player performance records
player_rows = []


# Loop through each matchup record
for _, row in matchups.iterrows():

    # Extract each player's scoring performance from the nested JSON column
    for player_id, points in row["players_points"].items():

        # Create one row per player performance
        player_rows.append({
            "season": row["season"],
            "week": row["week"],
            "roster_id": row["roster_id"],
            "matchup_id": row["matchup_id"],
            "player_id": player_id,
            "points": points
        })


# Convert flattened records into a dataframe
player_performance = pd.DataFrame(player_rows)


# Preview player performance fact table
player_performance.head()

StatementMeta(, 57e4cea9-aabd-413b-a396-cb683250fd09, 7, Finished, Available, Finished, False)

,season,week,roster_id,matchup_id,player_id,points
0,2024,1,1,6.0,10236,2.6
1,2024,1,1,6.0,11557,0.0
2,2024,1,1,6.0,11559,0.0
3,2024,1,1,6.0,11579,1.4
4,2024,1,1,6.0,11588,0.0


In [4]:
#check the row count

player_performance.shape

StatementMeta(, 57e4cea9-aabd-413b-a396-cb683250fd09, 9, Finished, Available, Finished, False)

(12312, 6)

In [8]:
# Create a unique roster identifier by combining season and roster ID.
# Roster IDs can repeat across seasons, so this key allows multiple years
# to be analyzed together without ambiguity.
player_performance["roster_key"] = (
    player_performance["season"].astype(str)
    + "_"
    + player_performance["roster_id"].astype(str)
)


# Preview player performance data with newly created roster key
player_performance.head()

StatementMeta(, 57e4cea9-aabd-413b-a396-cb683250fd09, 15, Finished, Available, Finished, False)

,season,week,roster_id,matchup_id,player_id,points,roster_key,season_week_key
0,2024,1,1,6.0,10236,2.6,2024_1,2024_01
1,2024,1,1,6.0,11557,0.0,2024_1,2024_01
2,2024,1,1,6.0,11559,0.0,2024_1,2024_01
3,2024,1,1,6.0,11579,1.4,2024_1,2024_01
4,2024,1,1,6.0,11588,0.0,2024_1,2024_01


In [9]:
# Create a unique season-week identifier for time-based analysis.
# This allows player performance records to connect to the calendar dimension
# and prevents duplicate week numbers across different seasons.
player_performance["season_week_key"] = (
    player_performance["season"].astype(str)
    + "_"
    + player_performance["week"].astype(str).str.zfill(2)
)

StatementMeta(, 57e4cea9-aabd-413b-a396-cb683250fd09, 17, Finished, Available, Finished, False)

In [10]:
#save as a delta table

spark_df = spark.createDataFrame(player_performance)

spark_df.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("Fact_Player_Performance")

StatementMeta(, 57e4cea9-aabd-413b-a396-cb683250fd09, 18, Finished, Available, Finished, True)